# Week 11 - Supersymmetric Collisions with deep FNN; CIFAR with CNNs

- Exercise 1 (recap from previous week): Fully connected networks for a physics problem: Is the collision supersymmetric or not?
- Exercise 2: We use a convolutional neural network to classify images from CIFAR and loko what is going on inside the network.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

# Exercise 1: Searching for Supersymmetric Collisions with Neural Nets

Today, we go back to an exercise, that we already started last week -- but most of you did not get to it.
Let's take some time to understand it properly, and review the code from last week!

**Feel free to refer to last weeks solutions for code reuse if you are unsure about your own.**

The line below downloads data from an existing experimental dataset for supersymmetric collisions.
For them to work you need to install `wget` and `gunzipz` -- if you do not have these installed you can directly go to the website below and download it via your webbrowser. 
Then you just need to unzip it (`.gz` is similar to `.zip`) - if you feel a bit lost then this is a good occasion to ask ChatGPT how to do it on your machine (or the TAs).

**If downloading is too slow, we recommend using google colab**

In [6]:
! wget https://archive.ics.uci.edu/ml/machine-learning-databases/00279/SUSY.csv.gz
! gunzip "/content/SUSY.csv.gz"

zsh:1: command not found: wget
gunzip: can't stat: /content/SUSY.csv.gz (/content/SUSY.csv.gz.gz): No such file or directory


The SUSY dataset consists of 5 million simulated Monte Carlo samples
of [supersymmetric and non-supersymmetric collisions](https://archive.ics.uci.edu/dataset/279/susy). The goal is to distinguish between a process
where new supersymmetric particles are produced and a background process. The first 8 features are
measurements of the final particle states, while the last 10 features are functions of the first 8 derived
by physicists to help to discriminate the events.

In [7]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets

We provide the class `SUSY_Dataset` to load the train and test datasets.
For many projects writing this class and cleaning the data is one of the most important steps in the deep learning pipeline.

In [8]:
class SUSY_Dataset(torch.utils.data.Dataset):
    """SUSY pytorch dataset."""

    def __init__(self, data_file, root_dir, dataset_size, train=True, transform=None, high_level_feats=None):
        """
        Args:
            data_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            dataset_size (int): Size of the full dataset which is splitted in 80% for train and 20% for test.
            train (bool, optional): If set to `True` load training data.
            transform (callable, optional): Optional transform to be applied on a sample.
            high_level_festures (bool, optional): If set to `True`, working with high-level features only.
                                        If set to `False`, working with low-level features only.
                                        Default is `None`: working with all features
        """

        import pandas as pd

        features=['SUSY','lepton 1 pT', 'lepton 1 eta', 'lepton 1 phi', 'lepton 2 pT', 'lepton 2 eta', 'lepton 2 phi',
                'missing energy magnitude', 'missing energy phi', 'MET_rel', 'axial MET', 'M_R', 'M_TR_2', 'R', 'MT2',
                'S_R', 'M_Delta_R', 'dPhi_r_b', 'cos(theta_r1)']

        low_features=['lepton 1 pT', 'lepton 1 eta', 'lepton 1 phi', 'lepton 2 pT', 'lepton 2 eta', 'lepton 2 phi',
                'missing energy magnitude', 'missing energy phi']

        high_features=['MET_rel', 'axial MET', 'M_R', 'M_TR_2', 'R', 'MT2','S_R', 'M_Delta_R', 'dPhi_r_b', 'cos(theta_r1)']

        # Number of datapoints to work with
        df = pd.read_csv(root_dir+data_file, header=None,nrows=dataset_size,engine='python')
        df.columns=features
        Y = df['SUSY']
        X = df[[col for col in df.columns if col!="SUSY"]]

        # Set training and test data size
        train_size=int(0.8*dataset_size)
        self.train=train

        if self.train:
            X=X[:train_size]
            Y=Y[:train_size]
            print("Training on {} examples".format(train_size))
        else:
            X=X[train_size:]
            Y=Y[train_size:]
            print("Testing on {} examples".format(dataset_size-train_size))

        self.root_dir = root_dir
        self.transform = transform

        # make datasets using only the 8 low-level features and 10 high-level features
        if high_level_feats is None:
            self.data=(X.values.astype(np.float32),Y.values.astype(int))
            print("Using both high and low level features")
        elif high_level_feats is True:
            self.data=(X[high_features].values.astype(np.float32),Y.values.astype(int))
            print("Using both high-level features only.")
        elif high_level_feats is False:
            self.data=(X[low_features].values.astype(np.float32),Y.values.astype(int))
            print("Using both low-level features only.")

    # override __len__ and __getitem__ of the Dataset() class

    def __len__(self):
        return len(self.data[1])

    def __getitem__(self, idx):

        sample=(self.data[0][idx,...],self.data[1][idx])

        if self.transform:
            sample=self.transform(sample)

        return sample

In [9]:
training_data = SUSY_Dataset(
    data_file='SUSY.csv',
    root_dir='./',
    dataset_size=2000,
    train=True,
    )

test_data = SUSY_Dataset(
    data_file='SUSY.csv',
    root_dir='./',
    dataset_size=2000,
    train=False,
    )

Training on 1600 examples
Using both high and low level features
Testing on 400 examples
Using both high and low level features


## Exercise 1.1

To train our neural network with SGD, we want to pass the samples in batches. Build the train and
test `data loaders`, setting the batch size to 100 and activating reshuffling at each epoch for the train
data by setting `shuffle=True`.

By shuffling the data we avoid periodicity in the training that comes from seeing all samples in the same order every time.

In [10]:
train_dataloader = ...
test_dataloader = ...

## Exercise 1.2

Some more words on the GPU:

Recall that in pytorch you have freedom where to put your model and your data -- on the CPU or on the GPU. 
You achieve this by setting `.to(device)` where `device` is either `'cpu'` or `'cuda'`, where the later means the framework that is running on your GPU (roughly).

It is good to have a basic idea on how the technology behind this works, but we do not require this for the course.
Basically your GPU has a given amount of storage (e.g. ~ 16GB, depends on how nice your GPU is), and on everything that is living on this storage (moved there via `.to('cuda')`) you can run efficient *parallel* computations.
This is similar to the vectorized efficient numpy functions you were using instead of for loops - only now you can parallelize even more. How many things you can parallelize depends on your specific GPU again, but adapting to that is handeled in the background by the pytorch framework. 

In any case - for the matrix-vector multiplications in the feedforward networks the GPU is super efficient, this is essentially what it is built for.

I our exercise you will not gain a lot by using the GPU, because our networks are very small.

Your laptop will likely not have a GPU, but if you upload this notebook to [google colab ](https://colab.research.google.com) you get free GPU time.
If no GPU is available, it is in practice still possible to execute every operation on a CPU - only slower. 

**We strongly recommend that you upload this notebook to colab if you experience problems with the speed of your computations (even their CPUs are quite fast).**

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu" # Checking whether a GPU is available to use, otherwise fall back to CPU
print(f"Using {device}")

Using cpu


Define a fully connected `ReLU` neural network taking an 18-dimensional input, with first two hidden layers with 200 neurons, one hidden layer with 100 neurons, and a final linear layer with two outputs neurons. You should use biases everywhere.

Do not forget to move your model to the correct device using `.to(device)`

In [17]:
model = ...

# Exercise 1.3

Using the cross-entropy loss and SGD with learning rate 1e-2, train the model. You can use the train and test loops defined in the previous exercise, as below, but adapt them to return the loss over the epochs to you and plot it.

- How many epochs do you need to converge on the training loss?
- On the test loss?
- Bonus: If you are curious, try to play with the architecture to get the test accuracy above 82%.
- Imagine you adapted the architecture according to the test accuracy and your friend comes along with a new fresh data sample from the same distribution. Under which circumstances would you expect the accuracy on the new sample to match/be lower/be higher the test accuracy of this best model you selected? 

In [19]:
def train(dataloader, model, loss_fn, optimizer):
    '''
    This function implements the train loop. It iterates over the training dataset
    and try to converge to optimal parameters.
    '''
    size = len(dataloader.dataset)
    model.train() # Set the model to training mode
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction and loss
        pred = model(X) # Pass the data to the model to execute the model forward
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward() # Compute gradients of the loss w.r.t parameters (backward pass)
        optimizer.step() # Do a gradient descent step and adjust parameters
        optimizer.zero_grad() # Reset the gradients of model parameters to zero (gradients by default add up)

        if batch % 200 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            
def test(dataloader, model, loss_fn):
    '''
    This function implements the validation/test loop. It iterates over the test
    dataset to check if the model performance is improving.
    '''
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() # Set the model to evaluation mode
    test_loss, correct = 0, 0
    with torch.no_grad(): # Do not track gradients while evaluating (faster)
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item() # Compute CE loss on the batch
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # Compute classification error
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
## YOUR TRAINING CODE HERE ##


# Exercise 2: Convolutional Neural Networks for CIFAR10


In this exercise, you will train a simple CNN to classify images from the CIFAR10 dataset.


## EXercise 2.1
Download the CIFAR10 dataset using `torchvision.datasets.CIFAR10`, and build the train and test dataloaders, setting the batch size to 32 and activating reshuffling at each epoch for the train data by setting `shuffle=True`. Visualize some images and their different color channels.
- What does the transform do?

In [22]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

training_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


train_dataloader = ...
test_dataloader = ...

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def imshow(img):
    img = img / 2 + 0.5 # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()
    
## YOUR CODE HERE ##
## VISUALIZE SOME IMAGES, of different classes ##

## Exercise 2.2

 Define a function returning a convolutional neural network built with `nn.Sequential`. Use a first layer of 6 convolutional channels with filter size 5, a max-pooling layer over a $2 \times 2$ window, a second convolutional layer made of 16 channels with filter size 5, another $2 \times 2$ max-pooling layer, two dense layers with 120 and 84 neurons respectively, and a final linear layer with 10 outputs.

 Use the `nn.Flatten()` operation if needed. You can also find convolutional layers in the pytorch documentation. 
 
 Bonus: What would be other ways to implement the same convolutional model?


In [27]:
def initialize_cnn():

    ### YOUR CODE ###

    return ...

## Exercise 2.3

Using the cross-entropy loss and SGD with learning rate 0.01, train the model for 5 epochs. After training, compute the accuracy on the test set. Did your algorithm converge?

Bonus: If you are on colab, use the GPU. Compare the times -- did it help speed the process up? Don't forget to also put the batches on the GPU.

In [ ]:
model = initialize_cnn()

## YOUR TRAINING CODE HERE ##

In [ ]:
## plot the loss curve ##

## Exercise 2.3 - Changing the optimizer

Making an analogy with a physical system, we can think of the negative gradient as a force moving a particle through parameter space, following Newton’s laws. Adding a momentum or inertia term, the optimization algorithm remembers the directions of the past gradients and continues to move in their direction. Mathematically,
$$
    v_t = \gamma v_{t-1} + \eta \nabla_\theta L(\theta_t)
$$
$$
    \theta_{t+1} = \theta_t - v_t,
$$
where $\gamma \in [0,1]$ is the momentum parameter, $\eta$ the learning rate, and $\theta$ the parameters of the model. Momentum helps the optimization dynamics gain speed in directions with persistent small gradients and suppresses oscillations. Repeat training, adding `momentum=0.9` to the SGD class.

- What is the effect of the momentum (compared to no momentum)? To understand, it helps to plot the loss curves.
- Bonus: Set the momentum value very high. What happens? Does that match your expectation?

In [ ]:
model = initialize_cnn()

## YOUR TRAINING CODE HERE ##

# Exercise 2.4 - Feature Maps

We often argue that neural networks are black-boxes. We say that we do not understand what is going on inside and how they do their computations.

In fact, they are not completely black-box: We can look inside and we know *every multiplication or addition* that is happening to produce the final prediction. We just do not know a useful way to **interpret** these introspections macroscopically - there is no simple explanation that arrises directly from the matrix-multiplications.

One way to get a better insight, is to understand how the input looks after a single layer or looks from the perspective of a single neuron. E.g. for which pixel is the neuron activated high and for which low? This allows us to get an intuition which **features** activate the neuron.

**A) - Layers** Using `torch.fx`, (**f**eature e**x**traction) we can visualize these transformations of an input inside our neural network. For different input images, check the outputs of the first convolutional layer, of the first ReLU application, and of the first pooling layer.

In [40]:
from torchvision.models.feature_extraction import get_graph_node_names
from torchvision.models.feature_extraction import create_feature_extractor

nodes, _ = get_graph_node_names(model)
print(nodes) # Prints the nn.Sequential layer names

feature_extractor = create_feature_extractor(
	model, return_nodes=['0', '1', '2']) # Outputs of first conv. layer, ReLU, and first pooling layer

['input', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']


In [ ]:
dataloader = DataLoader(training_data, batch_size=32)

dataiter = iter(dataloader)
images, labels = next(dataiter)
id = 4
image = images[id]

out = feature_extractor(image.unsqueeze(0))

imshow(image)

# show the outputs after the given layers

**B) - Neurons** Can we look at what a particular neuron reacts to? What are the features learned by deep models? A simple idea to visualize these features, called activation maximization, consists in looking for the input with bounded norm that maximizes the activation of a given neuron ($x^* = \arg \max_{x: \; \|x\|=1} h_i^{\ell}(x,\theta^*)$, where $h_i^\ell$ is the activation of the neuron $i$ at layer $\ell$ of a trained network). Open https://distill.pub/2017/feature-visualization/appendix and check how the neurons in different layers of the GoogLeNet network are specializing to recognize features with various complexity, from simple textures to meaningful semantic concepts!